# Filtering depth -- log-log over subset fraction

Holds `N = 100_000` fixed and sweeps the **subset fraction** so the
chunked-read benefit is visible directly.  Two panels:

| Panel | Geometry | Filter axis | Competitor |
| --- | --- | --- | --- |
| A | point cloud | bbox covering `f` of the cube volume | PLY (`plyfile`) |
| B | polyline / streamline | `f * N_polylines` random object_ids | TRX (`trx-python`, native partial read) |

`FRACTIONS = [0.001, 0.01, 0.1, 0.5, 1.0]` -- spans three decades on
the x-axis.  Log-log axes; shaded **95 % CI** band from `N_RUNS = 10`
repeats.

For the point-cloud panel zarr-vectors should slope down with `f`
(reading fewer chunks); PLY stays flat (it always parses the whole
file).  For the polyline panel both formats can do native partial
reads, so both should slope down -- the comparison there is "by how
much".

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    if not p.exists():
        return 0
    if p.is_file():
        return p.stat().st_size
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


def _new_file(prefix, suffix):
    """Fresh tempdir + competitor-format file path."""
    return Path(tempfile.mkdtemp(prefix=f'compbench_{prefix}_')) / f'data.{suffix}'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) -- hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)


def _repeat(fn, *args, n_runs=None, **kwargs):
    """Run fn n_runs times; return (mean_s, ci_hw_s)."""
    n = n_runs if n_runs is not None else N_RUNS
    samples = []
    for _ in range(n):
        t, _ = _time(fn, *args, **kwargs)
        samples.append(t)
    return _mean_ci95(samples)

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points, read_points
from zarr_vectors.types.polylines import write_polylines, read_polylines

N = 100_000
CHUNK = (200.0, 200.0, 200.0)
BIN   = (50.0, 50.0, 50.0)
SEED  = 0
EXTENT = 1000.0
FRACTIONS = [0.001, 0.01, 0.1, 0.5, 1.0]
rng = np.random.default_rng(SEED)


def _bbox_for_fraction(f):
    """Centred cube bbox covering fraction f of the EXTENT^3 volume."""
    hw = EXTENT * (f ** (1 / 3)) / 2
    centre = np.array([EXTENT / 2] * 3)
    return (centre - hw).tolist(), (centre + hw).tolist()

In [ ]:
# ----- PLY (point cloud) -----------------------------------------------------
def ply_write_points(path, positions):
    from plyfile import PlyData, PlyElement
    n = len(positions)
    arr = np.zeros(n, dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4')])
    arr['x'] = positions[:, 0]
    arr['y'] = positions[:, 1]
    arr['z'] = positions[:, 2]
    el = PlyElement.describe(arr, 'vertex')
    PlyData([el], text=False).write(str(path))


def ply_load_full(path):
    from plyfile import PlyData
    ply = PlyData.read(str(path))
    v = ply['vertex']
    return np.stack([v['x'], v['y'], v['z']], axis=1)


def ply_load_bbox(path, low, high):
    pts = ply_load_full(path)
    mask = np.all((pts >= np.asarray(low)) & (pts <= np.asarray(high)), axis=1)
    return pts[mask]


# ----- CSV edge-list (graph) -------------------------------------------------
def csv_write_graph(edges_path, nodes_path, positions, edges):
    pd.DataFrame({
        'node_id': np.arange(len(positions)),
        'x': positions[:, 0],
        'y': positions[:, 1],
        'z': positions[:, 2],
    }).to_csv(nodes_path, index=False)
    pd.DataFrame({
        'source': edges[:, 0],
        'target': edges[:, 1],
    }).to_csv(edges_path, index=False)


def csv_load_graph(edges_path, nodes_path):
    nodes = pd.read_csv(nodes_path)
    edges_df = pd.read_csv(edges_path)
    pos = nodes[['x', 'y', 'z']].to_numpy()
    e = edges_df[['source', 'target']].to_numpy()
    return pos, e


def csv_load_graph_bbox(edges_path, nodes_path, low, high):
    pos, e = csv_load_graph(edges_path, nodes_path)
    mask = np.all((pos >= np.asarray(low)) & (pos <= np.asarray(high)), axis=1)
    keep = np.where(mask)[0]
    keep_set = set(int(k) for k in keep)
    edge_mask = np.array([int(s) in keep_set and int(t) in keep_set for s, t in e])
    return pos[mask], e[edge_mask]


# ----- OBJ (mesh) -- pure-Python parser for fair comparison ------------------
def obj_write_mesh(path, vertices, faces):
    with open(path, 'w') as f:
        for v in vertices:
            f.write(f'v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n')
        for tri in faces:
            f.write(f'f {tri[0]+1} {tri[1]+1} {tri[2]+1}\n')


def obj_load_full(path):
    verts = []
    faces = []
    with open(path) as f:
        for line in f:
            if line.startswith('v '):
                parts = line.split()
                verts.append((float(parts[1]), float(parts[2]), float(parts[3])))
            elif line.startswith('f '):
                parts = line.split()
                tri = tuple(int(p.split('/')[0]) - 1 for p in parts[1:4])
                faces.append(tri)
    return np.asarray(verts, dtype=np.float32), np.asarray(faces, dtype=np.int64)


def obj_load_bbox(path, low, high):
    v, f = obj_load_full(path)
    mask = np.all((v >= np.asarray(low)) & (v <= np.asarray(high)), axis=1)
    keep = np.where(mask)[0]
    keep_set = set(int(k) for k in keep)
    face_mask = np.array([
        int(a) in keep_set and int(b) in keep_set and int(c) in keep_set
        for a, b, c in f
    ])
    return v[mask], f[face_mask]

In [ ]:
# TRX helpers (subset by object_ids).  Same shape as notebook 02.
def trx_write_polylines(path, polylines):
    import nibabel as nib
    from nibabel.streamlines import Tractogram
    from trx.trx_file_memmap import TrxFile, save as trx_save
    ref_img = nib.Nifti1Image(np.zeros((1, 1, 1), dtype=np.uint8), affine=np.eye(4))
    tractogram = Tractogram(streamlines=polylines, affine_to_rasmm=np.eye(4))
    trx_obj = TrxFile.from_tractogram(tractogram, reference=ref_img)
    trx_save(trx_obj, str(path))


def trx_load_objects(path, object_ids):
    from trx.trx_file_memmap import load as trx_load
    obj = trx_load(str(path))
    subset = obj.select(list(object_ids))
    return [np.asarray(s) for s in subset.streamlines]

## 2 · Build inputs

In [ ]:
# Points
pts = rng.uniform(0, EXTENT, (N, 3)).astype(np.float32)
zv_points  = _new_store('points')
ply_points = _new_file('points', 'ply')
write_points(zv_points, pts, chunk_shape=CHUNK, bin_shape=BIN)
ply_write_points(ply_points, pts)

# Polylines -- ~N total vertices spread across short walks.
counts = rng.integers(8, 16, size=N // 12)
polylines = []
for c in counts:
    start = rng.uniform(0, EXTENT, 3)
    steps = rng.normal(0, 5, (c, 3))
    polylines.append((start + steps.cumsum(axis=0)).astype(np.float32))
zv_polylines = _new_store('polylines')
write_polylines(zv_polylines, polylines, chunk_shape=CHUNK, bin_shape=BIN)
try:
    trx_polylines = _new_file('polylines', 'trx')
    trx_write_polylines(trx_polylines, polylines)
    trx_available = True
except Exception as e:
    print('TRX unavailable, polyline panel will be ZV-only:', e)
    trx_polylines = None
    trx_available = False

print(f'built: {len(pts):,} points, {len(polylines):,} polylines')

## 3 · Run the sweep

In [ ]:
n_poly = len(polylines)

# Pre-pick object id subsets per fraction so the same selection is
# used across all N_RUNS repeats.
poly_subsets = {
    f: list(rng.integers(0, n_poly, size=max(1, int(round(n_poly * f)))))
    for f in FRACTIONS
}

raw = {
    'points': {'zv': np.zeros((len(FRACTIONS), N_RUNS)),
               'comp': np.zeros((len(FRACTIONS), N_RUNS))},
    'polylines': {'zv': np.zeros((len(FRACTIONS), N_RUNS)),
                  'comp': np.zeros((len(FRACTIONS), N_RUNS))},
}

for i, f in enumerate(FRACTIONS):
    low, high = _bbox_for_fraction(f)
    ids = poly_subsets[f]

    # Points
    for run in range(N_RUNS):
        t_zv,   _ = _time(read_points, zv_points,
                          bbox=(low, high) if f < 1.0 else None)
        t_comp, _ = _time(
            ply_load_bbox if f < 1.0 else ply_load_full,
            ply_points,
            *((low, high) if f < 1.0 else ()),
        )
        raw['points']['zv'][i, run]   = t_zv
        raw['points']['comp'][i, run] = t_comp

    # Polylines
    for run in range(N_RUNS):
        t_zv, _ = _time(read_polylines, zv_polylines,
                        object_ids=None if f >= 1.0 else ids)
        if trx_available:
            from trx.trx_file_memmap import load as trx_load
            if f >= 1.0:
                t_comp, _ = _time(lambda: [np.asarray(s) for s in trx_load(str(trx_polylines)).streamlines])
            else:
                t_comp, _ = _time(trx_load_objects, trx_polylines, ids)
        else:
            t_comp = np.nan
        raw['polylines']['zv'][i, run]   = t_zv
        raw['polylines']['comp'][i, run] = t_comp

# Tidy long-form df.
rows = []
for panel in raw:
    for fmt in ('zv', 'comp'):
        for i, f in enumerate(FRACTIONS):
            samples = raw[panel][fmt][i]
            if np.any(np.isnan(samples)):
                mean, hw = np.nan, np.nan
            else:
                mean, hw = _mean_ci95(samples)
            rows.append({
                'panel':  panel,
                'fraction': f,
                'format': fmt,
                'mean_s': round(mean, 5) if mean == mean else None,
                'ci_hw':  round(hw, 5) if hw == hw else None,
            })

df = pd.DataFrame(rows)

## 4 · Results

In [ ]:
df

## 5 · Plot

In [ ]:
PANELS = [('points', 'point cloud (zv vs PLY)'),
          ('polylines', 'polylines (zv vs TRX)')]
SERIES = [('zarr-vectors', 'zv',   '#1f77b4'),
          ('competitor',   'comp', '#ff7f0e')]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
handles, labels = [], []

for ax, (panel, title) in zip(axes, PANELS):
    sub = df[df['panel'] == panel]
    for label, fmt, color in SERIES:
        s = sub[sub['format'] == fmt].sort_values('fraction').dropna(subset=['mean_s'])
        if s.empty:
            continue
        x    = s['fraction'].to_numpy()
        mean = s['mean_s'].to_numpy()
        hw   = s['ci_hw'].to_numpy()
        line, = ax.plot(x, mean, marker='o', color=color)
        ax.fill_between(x, mean - hw, mean + hw, color=color, alpha=0.20)
        if ax is axes[0]:
            handles.append(line)
            labels.append(label)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('subset fraction')
    ax.set_ylabel('s')
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)

fig.suptitle(f'Filter time vs subset fraction  --  N = {N:,}, {N_RUNS} runs, 95% CI')
fig.legend(handles, labels, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()